# Silver Layer - Trip Enrichment

## Objective

Transform raw trip records into a business-friendly dataset for analytics.

### Input Tables

- workspace.default.bronze_trips
- workspace.default.bronze_zones

### Transformations

- Standardize join keys
- Enrich pickup location
- Enrich dropoff location
- Calculate trip duration
- Apply data quality rules

### Output

- workspace.default.silver_trips


## Read Bronze Tables

Load trip and location reference data from the Bronze layer.

In [0]:
from pyspark.sql.functions import (
    col,
    unix_timestamp
)

bronze_trips = spark.table(
    "workspace.default.bronze_trips"
)

bronze_zones = spark.table(
    "workspace.default.bronze_zones"
)

## Standardize Join Keys

The trip table stores location identifiers as integers, while the zone table stores them as strings.

Convert both sides to the same datatype before joining.

In [0]:
zones_df = bronze_zones.withColumn(
    "LocationID",
    col("LocationID").cast("int")
)

## Location Enrichment

Join reference data to replace pickup and dropoff location IDs with meaningful zone information.

In [0]:
pickup_zones = zones_df.alias("pickup")

dropoff_zones = zones_df.alias("dropoff")

In [0]:
silver_df = (
    bronze_trips.alias("t")

    .join(
        pickup_zones,
        col("t.PULocationID") == col("pickup.LocationID"),
        "left"
    )

    .join(
        dropoff_zones,
        col("t.DOLocationID") == col("dropoff.LocationID"),
        "left"
    )

    .select(
        col("t.*"),

        col("pickup.Zone").alias("pickup_zone"),
        col("pickup.Borough").alias("pickup_borough"),

        col("dropoff.Zone").alias("dropoff_zone"),
        col("dropoff.Borough").alias("dropoff_borough")
    )
)

## Validation

Verify that pickup and dropoff locations have been successfully enriched.

In [0]:
display(
    silver_df.select(
        "PULocationID",
        "pickup_zone",
        "DOLocationID",
        "dropoff_zone"
    ).limit(20)
)

## Calculate Trip Duration

Create a derived metric representing total trip duration in minutes.

In [0]:
silver_df = silver_df.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp("tpep_dropoff_datetime")
        -
        unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

## Duration Profiling

Review duration statistics before applying quality rules.

In [0]:
silver_df.createOrReplaceTempView("silver_trip_view")

In [0]:
%sql
SELECT
    MIN(trip_duration_minutes) AS min_duration,
    MAX(trip_duration_minutes) AS max_duration,
    AVG(trip_duration_minutes) AS avg_duration
FROM silver_trip_view

## Apply Data Quality Rules

Remove records that are not suitable for analytical reporting.

Rules:

- fare_amount > 0
- trip_duration_minutes > 0
- trip_duration_minutes < 600

In [0]:
silver_clean_df = silver_df.filter(
    (col("fare_amount") > 0)
    &
    (col("trip_duration_minutes") > 0)
    &
    (col("trip_duration_minutes") < 600)
)

## Row Count Comparison

Measure the impact of Silver-layer cleansing.

In [0]:
print("Bronze Rows :", bronze_trips.count())
print("Silver Rows :", silver_clean_df.count())
print("Total rows dropped:",bronze_trips.count() - silver_clean_df.count())

## Sample Clean Records

Review the final Silver dataset before publishing.

In [0]:
display(
    silver_clean_df.select(
        "pickup_zone",
        "dropoff_zone",
        "fare_amount",
        "trip_duration_minutes"
    ).limit(20)
)

## Create Silver Delta Table

Persist the cleansed and enriched dataset for downstream analytics.

In [0]:
silver_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.silver_trips"
    )

## Validation

Confirm that the Silver table was created successfully.

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.silver_trips

In [0]:
%sql
SELECT
    MIN(trip_duration_minutes),
    MAX(trip_duration_minutes),
    AVG(trip_duration_minutes)
FROM workspace.default.silver_trips

In [0]:
%sql
SELECT
    pickup_borough,
    COUNT(*) trips
FROM workspace.default.silver_trips
GROUP BY pickup_borough
ORDER BY trips DESC

## Summary

### Silver Transformations

✅ Standardized join keys

✅ Enriched pickup locations

✅ Enriched dropoff locations

✅ Calculated trip duration

✅ Applied data quality rules

### Output Table

- workspace.default.silver_trips

### Next Step

Integrate a new source system using a Weather API and enrich trip data with weather conditions.

✅ Silver Layer Completed